# 12h — Explicit Dipole DIA Difference Cutouts + Dipole Fit

## Scientific motivation

The Rubin DIA difference image (`Difference` stamp delivered via Fink alerts) for a
stable Gaia star should ideally be a zero image.  In practice it carries residual
structure whose **morphology encodes the physical origin of the artefact**:

| Zernike mode | Noll index | Physical meaning in DIA context |
|---|---|---|
| Piston      | Z1  | Monopole — overall flux offset (photometric zero-point error, template flux bias) |
| Tip / Tilt  | Z2, Z3 | **Dipole** — centroid shift between science and template (astrometric offset, rotation) |
| Defocus     | Z4  | Symmetric ring — PSF size mismatch (seeing difference) |
| Astigmatism | Z5, Z6 | **Quadrupole** — PSF ellipticity mismatch |
| Coma        | Z7, Z8 | Lateral asymmetry — higher-order PSF wing asymmetry |
| Trefoil     | Z9, Z10 | Higher-order PSF shape |
| Spherical aberration | Z11 | Radially symmetric high-order PSF |

## Dipole fit methods implemented

| Method | Key | Description |
|---|---|---|
| Zernike decomposition | `zernike` | Project onto Noll basis; dipole from (c2, c3) |
| Linear gradient fit | `dipole_linear` | Image ≈ Ax·∂G/∂x + Ay·∂G/∂y + B (stable, fast) |
| Non-linear analytic dipole | `dipole_nl` | Two free Gaussians, least-squares |
| Shapelet decomposition | `shapelet` | Hermite basis; dipole from orders (1,0) and (0,1) |
| Explicit symmetric dipole | `dipole_explicit` | Two symmetric Gaussians with common sigma |
| Explicit asymmetric dipole | `dipole_asym` | Two independent-amplitude Gaussians + constraint |

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-17
- **Based on:** 12b_viewCutouts.ipynb
- **Subject:** Fink/LSST DIA — explicit dipole fitting and model comparison


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from astropy.time import Time
from math import factorial
from scipy.special import eval_hermite
from scipy.optimize import least_squares, minimize, NonlinearConstraint
from IPython.display import display as ipy_display

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
%matplotlib inline

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

objsid = {
    0: 170028527433285667,
    1: 170028526498480187,
    2: 170028485669027856,
    3: 313893022845108234,
    4: 170028526632697879,
    5: 170028529316003913,
}
DIAOBJECT_ID = objsid[1]

# Bands to analyse: None = all available, or e.g. ["r", "i"]
BANDS_FILTER = None

# Number of Zernike polynomials (Noll 1..N_ZERN).
# 15 covers up to radial order 4 (through spherical aberration).
N_ZERN = 15

# Number of Zernike terms to display in the per-visit bar-chart grid
N_DISPLAY = 11  # Z1..Z11

# Columns in the per-visit subplot grid
NCOLS_GRID = 4

# Image type to decompose: 'Difference' (alert-stream DIA diff)
CUTOUT_KIND = "Difference"

# Symmetric percentile for difference image display
DIFF_PERCENTILE = 99.0

# ─────────────────────────────────────────────────────────────────────────────
# METHOD SELECTION FLAG
# Choose which fit method to use for the final image triplet display
# (original / model / residuals).
# Valid values: "zernike" | "dipole_linear" | "dipole_nl" | "shapelet"
#               "dipole_explicit" | "dipole_asym"
# ─────────────────────────────────────────────────────────────────────────────
# SELECTED_METHOD = "dipole_linear"
SELECTED_METHOD = "dipole_explicit"

# Save figures?
SAVE_FIGS = True
DIR_FIGS = "figs_FINK_BLOCK_LC_12h"

# ─────────────────────────────────────────────────────────────────────────────
# Fixed paths
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_MANIFEST_FP = os.path.join(DIR_CUTOUTS, "manifest_fp.csv")
FILE_GAIA_MATCHED = os.path.join("data_FINK_BLOCK_LC_08b", "fink_diaobj_gaia_join_matched.csv")

AB_FLUX_ZERO = 3631e9
BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

# Noll-index labels for the first 21 Zernike polynomials
ZERNIKE_LABELS = {
    1: "Z1\npiston",
    2: "Z2\ntip (x)",
    3: "Z3\ntilt (y)",
    4: "Z4\ndefocus",
    5: "Z5\nastg 45°",
    6: "Z6\nastg 0°",
    7: "Z7\ncoma x",
    8: "Z8\ncoma y",
    9: "Z9\ntrefoil",
    10: "Z10\ntrefoil",
    11: "Z11\nspher",
    12: "Z12",
    13: "Z13",
    14: "Z14",
    15: "Z15",
    16: "Z16",
    17: "Z17",
    18: "Z18",
    19: "Z19",
    20: "Z20",
    21: "Z21",
}

# Mapping: method key → result dict keys for (recon, residual, quality, amp, angle)
METHOD_KEYS = {
    "zernike": ("recon_z", "residual_z", "quality_z", "dip_amp_z", "dip_angle_z"),
    "dipole_linear": ("recon_d_lin", "residual_d_lin", "quality_d_lin", "dip_amp_d_lin", "dip_angle_d_lin"),
    "dipole_nl": ("recon_d_nl", "residual_d_nl", "quality_d_nl", "dip_amp_d_nl", "dip_angle_d_nl"),
    "shapelet": ("recon_d_sh", "residual_d_sh", "quality_d_sh", "dip_amp_d_sh", "dip_angle_d_sh"),
    "dipole_explicit": ("recon_d_exp", "residual_d_exp", "quality_d_exp", "dip_amp_d_exp", "dip_angle_d_exp"),
    "dipole_asym": (
        "recon_d_asym",
        "residual_d_asym",
        "quality_d_asym",
        "dip_amp_d_asym",
        "dip_angle_d_asym",
    ),
}

assert SELECTED_METHOD in METHOD_KEYS, f"SELECTED_METHOD must be one of {list(METHOD_KEYS.keys())}"

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId   : {DIAOBJECT_ID}")
print(f"Manifest      : {os.path.abspath(FILE_MANIFEST)}")
print(f"Figures dir   : {os.path.abspath(DIR_FIGS)}")
print(f"N_ZERN        : {N_ZERN}   CUTOUT_KIND: {CUTOUT_KIND}")
print(f"Selected method for display: {SELECTED_METHOD}")

---
## 2. Fit model functions

All fitting functions share a common return contract:
```
params_or_coeffs, model_2d, residual_2d, quality_dict
```
The `quality` dict always contains:
- `r2`            : coefficient of determination R²
- `chi2_reduced`  : reduced chi² (sum(res²) / (N_pix - N_params))
- `rms_input`     : RMS of input pixel values
- `rms_residual`  : RMS of residual pixel values
- `dipole_amplitude` : scalar amplitude of the dipole component
- `dipole_angle_deg` : angle of the dipole vector in degrees (from +x axis)

---
### 2a. Shared PSF kernel helpers

In [ ]:
def gaussian_2d(x, y, x0, y0, sigma, amplitude=1.0):
    """Evaluate a 2-D isotropic Gaussian centred at (x0, y0)."""
    return amplitude * np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / (2.0 * sigma**2))


def _quality_metrics(img_valid, model_valid, n_params):
    """Compute R², chi2_reduced, rms_input, rms_residual from 1-D arrays.

    Parameters
    ----------
    img_valid   : 1-D array of pixel values inside the fit domain
    model_valid : 1-D array of model values at the same pixels
    n_params    : int, number of free parameters (degrees-of-freedom denominator)

    Returns
    -------
    dict with r2, chi2_reduced, rms_input, rms_residual
    """
    resid = img_valid - model_valid
    ss_res = float(np.sum(resid**2))
    ss_tot = float(np.sum((img_valid - np.mean(img_valid)) ** 2))
    n_pix = len(img_valid)
    dof = max(n_pix - n_params, 1)
    return {
        "r2": 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
        "chi2_reduced": ss_res / dof,
        "rms_input": float(np.sqrt(np.mean(img_valid**2))),
        "rms_residual": float(np.sqrt(np.mean(resid**2))),
    }


print("Shared PSF helpers ready.")

### 2b. Dipole linear fit  (`dipole_linear`)

Model: $I(x,y) \approx A_x\,\partial_x G + A_y\,\partial_y G + B$

where $G$ is a Gaussian centred on the image centroid.  The problem is linear
in $(A_x, A_y, B)$ → solved by `np.linalg.lstsq`.  Very stable, recommended
as the default.

- **Amplitude**: $\sqrt{A_x^2 + A_y^2}$
- **Angle**: $\text{atan2}(A_y, A_x)$ in degrees

In [ ]:
def fit_dipole_linear(image, sigma=1.5):
    """Fit a linear dipole model: I ≈ Ax·∂G/∂x + Ay·∂G/∂y + B.

    Parameters
    ----------
    image : (ny, nx) float array   — DIA difference cutout
    sigma : float                  — Gaussian width [pixels]

    Returns
    -------
    coeffs   : (3,) array  [Ax, Ay, B]
    model    : (ny, nx) array
    residual : (ny, nx) array
    quality  : dict with r2, chi2_reduced, rms_input, rms_residual,
                         dipole_amplitude, dipole_angle_deg
    """
    ny, nx = image.shape
    yy, xx = np.indices((ny, nx))

    valid = np.isfinite(image)
    x0 = np.mean(xx[valid])
    y0 = np.mean(yy[valid])

    # Gaussian and its partial derivatives, centred on image centroid
    G = np.exp(-((xx - x0) ** 2 + (yy - y0) ** 2) / (2.0 * sigma**2))
    dGdx = -(xx - x0) / sigma**2 * G
    dGdy = -(yy - y0) / sigma**2 * G

    A_mat = np.stack([dGdx[valid], dGdy[valid], np.ones(valid.sum())], axis=1)
    b = image[valid].astype(np.float64)

    coeffs, *_ = np.linalg.lstsq(A_mat, b, rcond=None)
    Ax, Ay, B = coeffs

    model = Ax * dGdx + Ay * dGdy + B
    residual = image - model

    # Dipole amplitude and angle
    dip_amp = float(np.hypot(Ax, Ay))
    dip_angle = float(np.degrees(np.arctan2(Ay, Ax)))

    quality = _quality_metrics(b, (A_mat @ coeffs), n_params=3)
    quality["dipole_amplitude"] = dip_amp
    quality["dipole_angle_deg"] = dip_angle

    return coeffs, model, residual, quality


print("fit_dipole_linear ready.")

### 2c. Non-linear analytic dipole  (`dipole_nl`)

Model: $I = A\,(G_+ - G_-) + B$  where the two Gaussian centres are free.
Parameters: $(x_0, y_0, dx, dy, A, B)$.
Non-linear optimisation via `least_squares` with `soft_l1` loss.

- **Amplitude**: $|A|$ (peak-to-peak / 2)
- **Angle**: $\text{atan2}(dy, dx)$ of the separation vector

In [ ]:
def _dipole_nl_model(params, xx, yy, sigma):
    """Evaluate the non-linear dipole model (internal helper)."""
    x0, y0, dx, dy, A, B = params
    g_pos = np.exp(-((xx - (x0 + dx)) ** 2 + (yy - (y0 + dy)) ** 2) / (2.0 * sigma**2))
    g_neg = np.exp(-((xx - (x0 - dx)) ** 2 + (yy - (y0 - dy)) ** 2) / (2.0 * sigma**2))
    return A * (g_pos - g_neg) + B


def fit_dipole_analytic(image, sigma=1.5):
    """Non-linear least-squares fit of a two-Gaussian dipole model.

    Parameters
    ----------
    image : (ny, nx) float array
    sigma : float  — fixed Gaussian width [pixels]

    Returns
    -------
    params   : (6,) array  [x0, y0, dx, dy, A, B]
    model    : (ny, nx) array
    residual : (ny, nx) array
    quality  : dict with r2, chi2_reduced, rms_input, rms_residual,
                         dipole_amplitude, dipole_angle_deg, dipole_separation
    """
    ny, nx = image.shape
    yy, xx = np.indices((ny, nx))

    valid = np.isfinite(image)
    x_data = xx[valid].astype(np.float64)
    y_data = yy[valid].astype(np.float64)
    z_data = image[valid].astype(np.float64)

    # Initial guess: gradient direction gives dipole orientation
    x0_g, y0_g = float(np.mean(x_data)), float(np.mean(y_data))
    gx = np.gradient(image, axis=1)
    gy = np.gradient(image, axis=0)
    dx0 = float(np.nanmean(gx[valid]))
    dy0 = float(np.nanmean(gy[valid]))
    norm = np.hypot(dx0, dy0) + 1e-8
    dx0 /= norm
    dy0 /= norm
    A0 = float(np.nanmax(image) - np.nanmin(image))
    B0 = float(np.nanmedian(image))

    p0 = [x0_g, y0_g, dx0, dy0, A0, B0]

    def residuals(p):
        return _dipole_nl_model(p, x_data, y_data, sigma) - z_data

    res = least_squares(residuals, p0, loss="soft_l1")
    p_opt = res.x
    x0, y0, dx, dy, A, B = p_opt

    model = _dipole_nl_model(p_opt, xx, yy, sigma)
    residual = image.astype(np.float64) - model

    # Dipole amplitude and angle
    dip_amp = float(np.abs(A))
    dip_angle = float(np.degrees(np.arctan2(dy, dx)))
    dip_sep = float(2.0 * np.hypot(dx, dy))

    quality = _quality_metrics(z_data, _dipole_nl_model(p_opt, x_data, y_data, sigma), n_params=6)
    quality["dipole_amplitude"] = dip_amp
    quality["dipole_angle_deg"] = dip_angle
    quality["dipole_separation"] = dip_sep

    return p_opt, model.astype(np.float32), residual.astype(np.float32), quality


print("fit_dipole_analytic ready.")

### 2d. Shapelet decomposition  (`shapelet`)

2-D Hermite-Gauss basis $\phi_{nm}(x,y) = H_n(x/\beta)\,H_m(y/\beta)\,e^{-(x^2+y^2)/(2\beta^2)}$.
Dipole components: orders $(1,0)$ (x-gradient) and $(0,1)$ (y-gradient).

- **Amplitude**: $\sqrt{c_{10}^2 + c_{01}^2}$
- **Angle**: $\text{atan2}(c_{01}, c_{10})$ in degrees

In [ ]:
def _shapelet_1d(n, x, beta):
    """1-D shapelet basis function of order n."""
    xi = x / beta
    return eval_hermite(n, xi) * np.exp(-0.5 * xi**2)


def _build_shapelet_basis(image_shape, n_max=4, beta=None):
    """Build 2-D shapelet design matrix."""
    ny, nx = image_shape
    yy, xx = np.indices((ny, nx))
    x = (xx - nx / 2.0).astype(np.float64)
    y = (yy - ny / 2.0).astype(np.float64)
    if beta is None:
        beta = 0.3 * min(nx, ny)
    valid = np.isfinite(x)  # all pixels (no NaN in coord grid)
    basis, orders = [], []
    for n in range(n_max + 1):
        for m in range(n_max + 1 - n):
            phi = _shapelet_1d(n, x, beta) * _shapelet_1d(m, y, beta)
            basis.append(phi[valid])
            orders.append((n, m))
    A = np.vstack(basis).T  # (n_pix, n_coeff)
    return A, orders, valid, float(beta)


def fit_shapelets(image, n_max=4, beta=None):
    """Fit a 2-D shapelet (Hermite-Gauss) decomposition.

    Parameters
    ----------
    image : (ny, nx) float array
    n_max : int   — maximum order (total basis functions ≈ (n_max+1)(n_max+2)/2)
    beta  : float — shapelet scale [pixels]; auto-selected if None

    Returns
    -------
    coeffs   : 1-D array
    model    : (ny, nx) array
    residual : (ny, nx) array
    quality  : dict with r2, chi2_reduced, rms_input, rms_residual,
                         dipole_amplitude, dipole_angle_deg, n_coeff
    orders   : list of (n,m) tuples matching coeffs
    """
    ny, nx = image.shape
    A, orders, valid, beta_used = _build_shapelet_basis(image.shape, n_max, beta)

    finite_mask = np.isfinite(image)
    b = image[finite_mask & valid].astype(np.float64)
    A_fit = A[finite_mask[valid], :]

    coeffs, *_ = np.linalg.lstsq(A_fit, b, rcond=None)

    model_vec = A @ coeffs
    model = np.full_like(image, np.nan, dtype=np.float64)
    model[valid] = model_vec
    residual = image.astype(np.float64) - model

    # Dipole amplitude and angle from shapelet orders (1,0) and (0,1)
    idx_map = {order: i for i, order in enumerate(orders)}
    Ax_sh = float(coeffs[idx_map[(1, 0)]]) if (1, 0) in idx_map else 0.0
    Ay_sh = float(coeffs[idx_map[(0, 1)]]) if (0, 1) in idx_map else 0.0
    dip_amp = float(np.hypot(Ax_sh, Ay_sh))
    dip_angle = float(np.degrees(np.arctan2(Ay_sh, Ax_sh)))

    quality = _quality_metrics(b, A_fit @ coeffs, n_params=len(coeffs))
    quality["dipole_amplitude"] = dip_amp
    quality["dipole_angle_deg"] = dip_angle
    quality["n_coeff"] = len(coeffs)
    quality["beta"] = beta_used

    return coeffs, model.astype(np.float32), residual.astype(np.float32), quality, orders


print("fit_shapelets ready.")

### 2e. Explicit symmetric dipole  (`dipole_explicit`)

Model: $I = F\,G(x_0+dx/2, y_0+dy/2) - F\,G(x_0-dx/2, y_0-dy/2) + B$

Parameters: $(x_0, y_0, dx, dy, F, B, \sigma)$ — 7 free parameters.

- **Amplitude**: $F \cdot \sqrt{dx^2 + dy^2}$ (product of flux × separation)
- **Angle**: $\text{atan2}(dy, dx)$ in degrees

In [ ]:
def _dipole_explicit_residuals(params, xx, yy, z_data):
    """Residual vector for the explicit symmetric dipole model."""
    x0, y0, dx, dy, F, B, sigma = params
    g_pos = gaussian_2d(xx, yy, x0 + dx / 2.0, y0 + dy / 2.0, sigma)
    g_neg = gaussian_2d(xx, yy, x0 - dx / 2.0, y0 - dy / 2.0, sigma)
    model = F * (g_pos - g_neg) + B
    return (model - z_data).ravel()


def fit_dipole_explicit(image, sigma_init=1.5):
    """Non-linear fit of an explicit symmetric two-Gaussian dipole.

    Parameters
    ----------
    image      : (ny, nx) float array
    sigma_init : float   — initial Gaussian width [pixels]

    Returns
    -------
    params   : (7,) array  [x0, y0, dx, dy, F, B, sigma]
    model    : (ny, nx) array
    residual : (ny, nx) array
    quality  : dict
    """
    ny, nx = image.shape
    yy, xx = np.indices((ny, nx), dtype=np.float64)
    valid = np.isfinite(image)
    x_data, y_data = xx[valid], yy[valid]
    z_data = image[valid].astype(np.float64)

    # Initial parameter guess
    x0_g, y0_g = float(np.mean(x_data)), float(np.mean(y_data))
    F_init = float((np.nanmax(image) - np.nanmin(image)) / 2.0)
    B_init = float(np.nanmedian(image))
    p0 = [x0_g, y0_g, 1.0, 1.0, F_init, B_init, sigma_init]

    # Bounds: F >= 0, sigma > 0
    lo = [-np.inf, -np.inf, -np.inf, -np.inf, 0.0, -np.inf, 0.01]
    hi = [np.inf] * 7

    result = least_squares(_dipole_explicit_residuals, p0, args=(x_data, y_data, z_data), bounds=(lo, hi))
    p_opt = result.x
    x0, y0, dx, dy, F, B, sigma = p_opt

    # Reconstruct full-image model
    g_pos = gaussian_2d(xx, yy, x0 + dx / 2.0, y0 + dy / 2.0, sigma)
    g_neg = gaussian_2d(xx, yy, x0 - dx / 2.0, y0 - dy / 2.0, sigma)
    model = F * (g_pos - g_neg) + B
    residual = image.astype(np.float64) - model

    # Dipole amplitude and angle
    dip_amp = float(np.abs(F) * np.hypot(dx, dy))
    dip_angle = float(np.degrees(np.arctan2(dy, dx)))

    model_valid = model[valid]
    quality = _quality_metrics(z_data, model_valid, n_params=7)
    quality["dipole_amplitude"] = dip_amp
    quality["dipole_angle_deg"] = dip_angle
    quality["dipole_separation"] = float(np.hypot(dx, dy))

    return p_opt, model.astype(np.float32), residual.astype(np.float32), quality


print("fit_dipole_explicit ready.")

### 2f. Explicit asymmetric dipole  (`dipole_asym`)

Model: $I = F_+\,G_+ + F_-\,G_- + B$  with $F_+$ and $F_-$ independent
(8 free parameters).
An optional non-linear constraint enforces $|F_+ - F_-|\,/\,(|F_+|+|F_-|) < 0.65$
(LSST dipole validity criterion).

- **Amplitude**: $(|F_+| + |F_-|) / 2$
- **Angle**: $\text{atan2}(dy, dx)$ of the lobe separation

In [ ]:
def fit_dipole_with_asymmetry(image, sigma_init=1.5, enforce_flux_constraint=True):
    """Non-linear fit of an asymmetric two-Gaussian dipole.

    Parameters
    ----------
    image                   : (ny, nx) float array
    sigma_init              : float  — initial Gaussian width [pixels]
    enforce_flux_constraint : bool   — if True, require flux ratio < 0.65

    Returns
    -------
    params   : (8,) array  [x0, y0, dx, dy, F_pos, F_neg, B, sigma]
    model    : (ny, nx) array
    residual : (ny, nx) array
    quality  : dict
    """
    ny, nx = image.shape
    yy, xx = np.indices((ny, nx), dtype=np.float64)
    valid = np.isfinite(image)
    x_data, y_data = xx[valid], yy[valid]
    z_data = image[valid].astype(np.float64)

    x0_g, y0_g = float(np.mean(x_data)), float(np.mean(y_data))
    F_pos_init = float(np.nanmax(image))
    F_neg_init = float(np.nanmin(image))
    B_init = float(np.nanmedian(image))
    p0 = [x0_g, y0_g, 1.0, 1.0, F_pos_init, F_neg_init, B_init, sigma_init]

    def objective(p):
        x0, y0, dx, dy, Fp, Fn, B, sigma = p
        g_pos = gaussian_2d(x_data, y_data, x0 + dx / 2.0, y0 + dy / 2.0, max(sigma, 0.01))
        g_neg = gaussian_2d(x_data, y_data, x0 - dx / 2.0, y0 - dy / 2.0, max(sigma, 0.01))
        return np.sum((Fp * g_pos + Fn * g_neg + B - z_data) ** 2)

    bounds = [(-np.inf, np.inf)] * 8
    bounds[4] = (0.0, np.inf)  # F_pos > 0
    bounds[5] = (-np.inf, 0.0)  # F_neg < 0
    bounds[7] = (0.01, np.inf)  # sigma > 0

    optim_kwargs = dict(method="SLSQP", bounds=bounds)
    if enforce_flux_constraint:

        def flux_ratio_constraint(p):
            Fp, Fn = p[4], p[5]
            return 0.65 - np.abs(Fp - Fn) / (np.abs(Fp) + np.abs(Fn) + 1e-30)

        optim_kwargs["constraints"] = [{"type": "ineq", "fun": flux_ratio_constraint}]

    result = minimize(objective, p0, **optim_kwargs)
    p_opt = result.x
    x0, y0, dx, dy, Fp, Fn, B, sigma = p_opt
    sigma = max(sigma, 0.01)

    g_pos = gaussian_2d(xx, yy, x0 + dx / 2.0, y0 + dy / 2.0, sigma)
    g_neg = gaussian_2d(xx, yy, x0 - dx / 2.0, y0 - dy / 2.0, sigma)
    model = Fp * g_pos + Fn * g_neg + B
    residual = image.astype(np.float64) - model

    # Dipole amplitude and angle
    dip_amp = float((np.abs(Fp) + np.abs(Fn)) / 2.0)
    dip_angle = float(np.degrees(np.arctan2(dy, dx)))
    flux_ratio = float(np.abs(Fp - Fn) / (np.abs(Fp) + np.abs(Fn) + 1e-30))

    model_valid = model[valid]
    quality = _quality_metrics(z_data, model_valid, n_params=8)
    quality["dipole_amplitude"] = dip_amp
    quality["dipole_angle_deg"] = dip_angle
    quality["flux_ratio"] = flux_ratio
    quality["is_dipole_lsst"] = bool(flux_ratio < 0.65)

    return p_opt, model.astype(np.float32), residual.astype(np.float32), quality


print("fit_dipole_with_asymmetry ready.")

---
## 3. Zernike polynomial engine

Noll (1976) sequential indexing. Polynomials computed from scratch using
NumPy/SciPy — no external Zernike library required.

In [ ]:
def noll_to_nm(j):
    """Convert Noll sequential index j (1-based) to (n, m) following Noll (1976)."""
    if j < 1:
        raise ValueError("j must be >= 1")
    count, n = 1, 0
    while True:
        m_list = [0] if n % 2 == 0 else []
        step = 2 if n % 2 == 0 else 1
        start = 2 if n % 2 == 0 else 1
        for m in range(start, n + 1, 2):
            m_list.extend([m, m])
        if n % 2 != 0:
            m_list = []
            for m in range(1, n + 1, 2):
                m_list.extend([m, m])
        for m_abs in m_list:
            if count == j:
                if m_abs == 0:
                    return n, 0
                return (n, m_abs) if count % 2 == 0 else (n, -m_abs)
            count += 1
        n += 1


# Validate against Noll 1976 Table 1
_NOLL_EXPECTED = {
    1: (0, 0),
    2: (1, 1),
    3: (1, -1),
    4: (2, 0),
    5: (2, -2),
    6: (2, 2),
    7: (3, -1),
    8: (3, 1),
    9: (3, -3),
    10: (3, 3),
    11: (4, 0),
}
for _j, _expected in _NOLL_EXPECTED.items():
    _got = noll_to_nm(_j)
    assert _got == _expected, f"noll_to_nm({_j}) = {_got}, expected {_expected}"
print("Noll index conversion validated against Table 1.")


def zernike_radial(n, m_abs, rho):
    """Radial Zernike polynomial R_n^m(rho) via direct summation."""
    m = m_abs
    result = np.zeros_like(rho, dtype=np.float64)
    for s in range((n - m) // 2 + 1):
        coeff = (
            (-1) ** s
            * factorial(n - s)
            / (factorial(s) * factorial((n + m) // 2 - s) * factorial((n - m) // 2 - s))
        )
        result += coeff * rho ** (n - 2 * s)
    return result


def zernike_poly(j, rho, theta):
    """Evaluate the j-th Zernike polynomial (Noll, orthonormal on unit disk)."""
    n, m = noll_to_nm(j)
    R = zernike_radial(n, abs(m), rho)
    if m == 0:
        return np.sqrt(n + 1) * R
    elif m > 0:
        return np.sqrt(2 * (n + 1)) * R * np.cos(m * theta)
    else:
        return np.sqrt(2 * (n + 1)) * R * np.sin(abs(m) * theta)


print("Zernike polynomial engine ready.")

## 4. Zernike basis matrix builder and fit function

In [ ]:
_ZERNIKE_CACHE = {}


def build_zernike_basis(ny, nx, n_zern):
    """Build the Zernike design matrix for a (ny, nx) pixel grid (cached).

    Returns
    -------
    A     : (n_in_disk, n_zern) — design matrix
    mask  : (ny, nx) bool       — True inside the inscribed unit disk
    rho   : (ny, nx) array      — normalised radius [0, 1]
    theta : (ny, nx) array      — azimuthal angle [rad]
    """
    key = (ny, nx, n_zern)
    if key in _ZERNIKE_CACHE:
        return _ZERNIKE_CACHE[key]

    cy, cx = (ny - 1) / 2.0, (nx - 1) / 2.0
    yy, xx = np.mgrid[0:ny, 0:nx]
    dy = yy - cy
    dx = xx - cx
    r_max = min(ny, nx) / 2.0
    rho = np.sqrt(dx**2 + dy**2) / r_max
    theta = np.arctan2(-dy, dx)  # Noll sign convention
    mask = rho <= 1.0

    rho_in, theta_in = rho[mask], theta[mask]
    n_in = int(mask.sum())
    A = np.zeros((n_in, n_zern), dtype=np.float64)
    for j in range(1, n_zern + 1):
        A[:, j - 1] = zernike_poly(j, rho_in, theta_in)

    _ZERNIKE_CACHE[key] = (A, mask, rho, theta)
    print(f"  Built Zernike basis: {ny}×{nx}  n_in_disk={n_in}  n_zern={n_zern}")
    return A, mask, rho, theta


def fit_zernike(image, n_zern, weights=None):
    """Project a 2-D image onto the first n_zern Zernike polynomials (Noll).

    Parameters
    ----------
    image   : (ny, nx) float array
    n_zern  : int
    weights : (ny, nx) array or None   — per-pixel weights (e.g. 1/sigma²)

    Returns
    -------
    coeffs  : (n_zern,) array
    recon   : (ny, nx) array  — reconstructed image (NaN outside disk)
    residual: (ny, nx) array
    quality : dict with r2, chi2_reduced, rms_input, rms_residual,
                        dipole_amplitude, dipole_angle_deg, n_fit_pixels
    """
    ny, nx = image.shape
    A, disk_mask, rho, theta = build_zernike_basis(ny, nx, n_zern)

    # Combined validity mask: inside disk AND finite
    valid_2d = np.isfinite(image) & disk_mask
    valid_in = valid_2d[disk_mask]  # boolean, size = n_in_disk
    A_fit = A[valid_in, :]
    img_fit = image[disk_mask][valid_in].astype(np.float64)

    if weights is not None:
        sqrt_w = np.sqrt(np.maximum(weights[disk_mask][valid_in].astype(np.float64), 0.0))
        coeffs, *_ = np.linalg.lstsq(A_fit * sqrt_w[:, None], img_fit * sqrt_w, rcond=None)
    else:
        coeffs, *_ = np.linalg.lstsq(A_fit, img_fit, rcond=None)

    # Reconstruct on the full disk grid
    recon_2d = np.full((ny, nx), np.nan, dtype=np.float64)
    recon_2d[disk_mask] = A @ coeffs
    residual_2d = image.astype(np.float64) - recon_2d

    # Dipole amplitude and angle from Z2 (tip) and Z3 (tilt)
    c2 = float(coeffs[1]) if n_zern >= 2 else 0.0
    c3 = float(coeffs[2]) if n_zern >= 3 else 0.0
    dip_amp = float(np.hypot(c2, c3))
    dip_angle = float(np.degrees(np.arctan2(c3, c2)))

    quality = _quality_metrics(img_fit, A_fit @ coeffs, n_params=n_zern)
    quality["dipole_amplitude"] = dip_amp
    quality["dipole_angle_deg"] = dip_angle
    quality["n_fit_pixels"] = int(valid_in.sum())

    return coeffs, recon_2d.astype(np.float32), residual_2d.astype(np.float32), quality


def zernike_group_power(coeffs):
    """Compute the power (sum of squared coefficients) per morphological group."""
    groups = {
        "Z1_piston": [1],
        "Z2-3_dipole": [2, 3],
        "Z4_defocus": [4],
        "Z5-6_astig": [5, 6],
        "Z7-8_coma": [7, 8],
        "Z9-10_trefoil": [9, 10],
        "Z11_spher": [11],
        "Z12+_higher": list(range(12, len(coeffs) + 1)),
    }
    return {
        name: float(np.sum(coeffs[np.array([i for i in indices if i <= len(coeffs)]) - 1] ** 2))
        for name, indices in groups.items()
    }


print("Zernike basis builder and fit_zernike ready.")

---
## 5. Utility functions

In [ ]:
def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


def parse_bool(val):
    """Coerce a dipole flag value to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def load_npy(src_id, band, kind):
    """Load fullcutouts_{obj}/cutouts/{src_id}_{band}_{kind}.npy → float32 or None."""
    fpath = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.npy")
    return np.load(fpath).astype(np.float32) if os.path.exists(fpath) else None


def symvlim(arr, percentile=99.0):
    """Symmetric half-range for a diverging colour scale."""
    finite = arr[np.isfinite(arr)]
    return max(float(np.percentile(np.abs(finite), percentile)), 1e-9) if len(finite) else 1e-9


def draw_crosshair(ax, shape):
    """Draw dashed yellow crosshair at image centre."""
    ny, nx = shape
    ax.axvline(nx / 2, color="yellow", lw=0.8, ls="--", alpha=0.7)
    ax.axhline(ny / 2, color="yellow", lw=0.8, ls="--", alpha=0.7)


print("Utility functions ready.")

---
## 6. Load manifest & Gaia metadata

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\nRun: python fink_download_full_cutouts.py --obj_id {DIAOBJECT_ID}"
    )

df = pd.read_csv(FILE_MANIFEST)
df["isDipole"] = df["r:isDipole"].fillna(False).apply(parse_bool)
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
print(f"Manifest : {len(df)} diaSources   bands: {sorted(df['r:band'].unique())}")

# Gaia metadata
gaia_G_mag, gaia_status, fink_group = np.nan, "?", "?"
if os.path.exists(FILE_GAIA_MATCHED):
    dg = pd.read_csv(FILE_GAIA_MATCHED)
    hit = dg[dg["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        gaia_G_mag = float(r.get("gaia_phot_g_mean_mag", np.nan))
        gaia_status = str(r.get("gaia_status", "?"))
        fink_group = str(r.get("group", "?"))
G_str = f"{gaia_G_mag:.2f}" if np.isfinite(gaia_G_mag) else "?"
print(f"Gaia G={G_str}  status={gaia_status}  group={fink_group}")

bands_available = [b for b in BAND_ORDER if b in df["r:band"].values]
bands_to_use = [b for b in bands_available if BANDS_FILTER is None or b in BANDS_FILTER]
print(f"Bands to analyse : {bands_to_use}")

---
## 7. Run all fit methods — collect results per visit

For each diaSource cutout, all six methods are run and their outputs stored
in `results[band]` — a list of dicts, one per valid visit.

In [ ]:
results = {}  # results[band] = list of dicts

for band in bands_to_use:
    df_b = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)
    print(f"\n{'─' * 60}")
    print(f"  Band {band}  —  {len(df_b)} diaSources")
    print(f"{'─' * 60}")

    band_results = []
    for src_idx, (_, row) in enumerate(df_b.iterrows(), start=1):
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        mjd = float(row["r:midpointMjdTai"])
        date = mjd_to_date(mjd)
        is_dip = bool(row["isDipole"])
        psf_f = float(row.get("r:psfFlux", np.nan))
        snr = float(row.get("r:snr", np.nan))

        arr = load_npy(src_id, band, CUTOUT_KIND)
        if arr is None:
            print(f"  [{src_idx:02d}] visit={visit}  {date}  → {CUTOUT_KIND} missing, skip")
            continue

        # ── Zernike decomposition ──────────────────────────────────────────
        coeffs_z, recon_z, residual_z, quality_z = fit_zernike(arr, N_ZERN)
        power_z = zernike_group_power(coeffs_z)  # uses coeffs_z (bug fix)

        # ── Non-linear analytic dipole ─────────────────────────────────────
        p_nl, recon_nl, residual_nl, quality_nl = fit_dipole_analytic(arr)

        # ── Linear gradient dipole ─────────────────────────────────────────
        lin_coeffs, recon_lin, residual_lin, quality_lin = fit_dipole_linear(arr)

        # ── Shapelet decomposition ─────────────────────────────────────────
        sh_coeffs, recon_sh, residual_sh, quality_sh, sh_orders = fit_shapelets(arr)

        # ── Explicit symmetric dipole ──────────────────────────────────────
        p_exp, recon_exp, residual_exp, quality_exp = fit_dipole_explicit(arr)

        # ── Explicit asymmetric dipole ─────────────────────────────────────
        p_asym, recon_asym, residual_asym, quality_asym = fit_dipole_with_asymmetry(arr)

        rec = {
            # Metadata
            "src_id": src_id,
            "visit": visit,
            "mjd": mjd,
            "date": date,
            "is_dipole": is_dip,
            "psfFlux": psf_f,
            "snr": snr,
            "arr": arr,
            # ── Zernike
            "recon_z": recon_z,
            "residual_z": residual_z,
            "z_coeffs": coeffs_z,
            "z_power": power_z,
            "quality_z": quality_z,
            "dip_amp_z": quality_z["dipole_amplitude"],
            "dip_angle_z": quality_z["dipole_angle_deg"],
            # ── Dipole non-linear
            "recon_d_nl": recon_nl,
            "residual_d_nl": residual_nl,
            "quality_d_nl": quality_nl,
            "dip_amp_d_nl": quality_nl["dipole_amplitude"],
            "dip_angle_d_nl": quality_nl["dipole_angle_deg"],
            # ── Dipole linear
            "recon_d_lin": recon_lin,
            "residual_d_lin": residual_lin,
            "quality_d_lin": quality_lin,
            "dip_amp_d_lin": quality_lin["dipole_amplitude"],
            "dip_angle_d_lin": quality_lin["dipole_angle_deg"],
            # ── Shapelet
            "recon_d_sh": recon_sh,
            "residual_d_sh": residual_sh,
            "sh_coeffs": sh_coeffs,
            "quality_d_sh": quality_sh,
            "dip_amp_d_sh": quality_sh["dipole_amplitude"],
            "dip_angle_d_sh": quality_sh["dipole_angle_deg"],
            # ── Explicit symmetric
            "recon_d_exp": recon_exp,
            "residual_d_exp": residual_exp,
            "quality_d_exp": quality_exp,
            "dip_amp_d_exp": quality_exp["dipole_amplitude"],
            "dip_angle_d_exp": quality_exp["dipole_angle_deg"],
            # ── Explicit asymmetric
            "recon_d_asym": recon_asym,
            "residual_d_asym": residual_asym,
            "quality_d_asym": quality_asym,
            "dip_amp_d_asym": quality_asym["dipole_amplitude"],
            "dip_angle_d_asym": quality_asym["dipole_angle_deg"],
        }
        band_results.append(rec)

        dip_flag = "  🔴 DIPOLE" if is_dip else ""
        print(
            f"  [{src_idx:02d}] visit={visit}  {date}  shape={arr.shape}{dip_flag}\n"
            f"    Zernike  : R²={quality_z['r2']:.3f}  χ²r={quality_z['chi2_reduced']:.3f}  "
            f"A={quality_z['dipole_amplitude']:.2f}  θ={quality_z['dipole_angle_deg']:.1f}°\n"
            f"    Dip NL   : R²={quality_nl['r2']:.3f}  χ²r={quality_nl['chi2_reduced']:.3f}  "
            f"A={quality_nl['dipole_amplitude']:.2f}  θ={quality_nl['dipole_angle_deg']:.1f}°\n"
            f"    Dip LIN  : R²={quality_lin['r2']:.3f}  χ²r={quality_lin['chi2_reduced']:.3f}  "
            f"A={quality_lin['dipole_amplitude']:.2f}  θ={quality_lin['dipole_angle_deg']:.1f}°\n"
            f"    Shapelet : R²={quality_sh['r2']:.3f}  χ²r={quality_sh['chi2_reduced']:.3f}  "
            f"A={quality_sh['dipole_amplitude']:.2f}  θ={quality_sh['dipole_angle_deg']:.1f}°\n"
            f"    Dip EXP  : R²={quality_exp['r2']:.3f}  χ²r={quality_exp['chi2_reduced']:.3f}  "
            f"A={quality_exp['dipole_amplitude']:.2f}  θ={quality_exp['dipole_angle_deg']:.1f}°\n"
            f"    Dip ASYM : R²={quality_asym['r2']:.3f}  χ²r={quality_asym['chi2_reduced']:.3f}  "
            f"A={quality_asym['dipole_amplitude']:.2f}  θ={quality_asym['dipole_angle_deg']:.1f}°"
        )

    results[band] = band_results
    print(f"  → {len(band_results)} valid fits for band {band}")

print("\nAll fits complete.")

---
## 8. Method comparison summary table

For each visit, compare all six methods on R², reduced χ², dipole amplitude,
dipole angle, and rms_residual.  The **best method per visit** (highest R²)
is highlighted in green.

In [ ]:
METHODS_DISPLAY = [
    ("zernike", "quality_z"),
    ("dipole_linear", "quality_d_lin"),
    ("dipole_nl", "quality_d_nl"),
    ("shapelet", "quality_d_sh"),
    ("dipole_explicit", "quality_d_exp"),
    ("dipole_asym", "quality_d_asym"),
]

rows = []
for band in bands_to_use:
    for rec in results.get(band, []):
        for method_key, quality_key in METHODS_DISPLAY:
            q = rec[quality_key]
            rows.append(
                {
                    "band": band,
                    "visit": rec["visit"],
                    "date": rec["date"],
                    "isDipole": rec["is_dipole"],
                    "method": method_key,
                    "R2": round(float(q.get("r2", np.nan)), 4),
                    "chi2_red": round(float(q.get("chi2_reduced", np.nan)), 4),
                    "rms_resid": round(float(q.get("rms_residual", np.nan)), 4),
                    "dip_amp": round(float(q.get("dipole_amplitude", np.nan)), 4),
                    "dip_angle": round(float(q.get("dipole_angle_deg", np.nan)), 2),
                }
            )

df_cmp = pd.DataFrame(rows)


df_cmp["is_best"] = df_cmp.groupby(["band", "visit"])["R2"].transform(lambda x: x == x.max())


def highlight_best_r2(row):
    return [
        "background-color: #d4edda; font-weight: bold" if row["is_best"] and col == "R2" else ""
        for col in row.index
    ]


styled = df_cmp.style.apply(highlight_best_r2, axis=1).format(
    {
        "R2": "{:.4f}",
        "chi2_red": "{:.4f}",
        "rms_resid": "{:.4f}",
        "dip_amp": "{:.4f}",
        "dip_angle": "{:.2f}",
    }
)


print(
    f"Comparison table: {len(df_cmp)} rows  ({len(df_cmp) // len(METHODS_DISPLAY)} visits × {len(METHODS_DISPLAY)} methods)"
)
ipy_display(styled)

csv_path = os.path.join(DIR_FIGS, f"method_comparison_obj{DIAOBJECT_ID}.csv")
df_cmp.to_csv(csv_path, index=False)
print(f"→ Saved to {csv_path}")

### 8b. Best method per visit (pivot view)

In [ ]:
# Pivot: rows = visits, columns = methods, values = R²
df_pivot_r2 = df_cmp.pivot_table(
    index=["band", "visit", "date", "isDipole"], columns="method", values="R2"
).round(4)

df_pivot_chi2 = df_cmp.pivot_table(
    index=["band", "visit", "date", "isDipole"], columns="method", values="chi2_red"
).round(4)

df_pivot_r2["best_method"] = df_pivot_r2[[m for m, _ in METHODS_DISPLAY]].idxmax(axis=1)

print("R² pivot (rows = visits, cols = methods):")
ipy_display(
    df_pivot_r2.style.highlight_max(axis=1, subset=[m for m, _ in METHODS_DISPLAY], color="#d4edda")
    .format("{:.4f}", subset=[m for m, _ in METHODS_DISPLAY])
    .set_caption("R² by method and visit — higher is better")
)

print("\nχ²_red pivot:")
ipy_display(
    df_pivot_chi2.style.highlight_min(axis=1, subset=[m for m, _ in METHODS_DISPLAY], color="#cce5ff")
    .format("{:.4f}", subset=[m for m, _ in METHODS_DISPLAY])
    .set_caption("χ²_red by method and visit — lower is better (ideal ≈ 1)")
)

print(f"\nCurrently selected display method: {SELECTED_METHOD}")
best_counts = df_pivot_r2["best_method"].value_counts()
print("\nMethod winning most visits (by R²):")
print(best_counts.to_string())

---
## 9. Figure A — Zernike coefficient spectrum (all visits, one plot per band)

In [ ]:
jj = np.arange(1, N_ZERN + 1)
xlabels = [ZERNIKE_LABELS.get(j, f"Z{j}") for j in jj]

group_spans = [
    (1, 1, "#f0e0ff", "piston"),
    (2, 3, "#ffe0e0", "dipole"),
    (4, 4, "#e0f0ff", "defocus"),
    (5, 6, "#e0ffe0", "astig"),
    (7, 8, "#fff0e0", "coma"),
    (9, 10, "#f0ffe0", "trefoil"),
    (11, 11, "#e8e8ff", "spher"),
]

for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    mjds = np.array([r["mjd"] for r in band_res])
    norm = mcolors.Normalize(vmin=mjds.min(), vmax=max(mjds.max(), mjds.min() + 1))
    cmap_v = cm.get_cmap("plasma")

    fig, (ax_abs, ax_signed) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"hspace": 0.35})
    for rec in band_res:
        c = cmap_v(norm(rec["mjd"]))
        lw = 1.8 if rec["is_dipole"] else 0.9
        ls = "-" if rec["is_dipole"] else "--"
        ax_abs.semilogy(jj, np.abs(rec["z_coeffs"]) + 1e-6, color=c, lw=lw, ls=ls, alpha=0.85)
        ax_signed.plot(jj, rec["z_coeffs"], color=c, lw=lw, ls=ls, alpha=0.85)

    sm = cm.ScalarMappable(cmap=cmap_v, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax_abs, ax_signed], fraction=0.025, pad=0.02)
    cbar.set_label("MJD", fontsize=9)

    for j0, j1, fc, lbl in group_spans:
        if j1 > N_ZERN:
            continue
        for ax in (ax_abs, ax_signed):
            ax.axvspan(j0 - 0.5, j1 + 0.5, color=fc, alpha=0.35, zorder=0)
        ax_abs.text(
            (j0 + j1) / 2,
            1e-4,
            lbl,
            ha="center",
            va="bottom",
            fontsize=7,
            color="#555",
            transform=ax_abs.get_xaxis_transform(),
        )

    ax_abs.set_ylabel("|c_j|  (log)", fontsize=9)
    ax_abs.set_title(
        f"Zernike |coefficient| spectrum — band {band} — {len(band_res)} visits", fontsize=10, color=bcolor
    )
    ax_abs.grid(True, which="both", alpha=0.3)
    ax_signed.axhline(0, color="k", lw=0.7, ls=":")
    ax_signed.set_ylabel("c_j  (signed)", fontsize=9)
    ax_signed.set_xticks(jj)
    ax_signed.set_xticklabels(xlabels, fontsize=7, rotation=0)
    ax_signed.set_xlabel("Noll index", fontsize=9)
    ax_signed.grid(True, alpha=0.3)

    fig.suptitle(
        f"diaObjectId={DIAOBJECT_ID}   band={band}   Gaia G={G_str}  status={gaia_status}  group={fink_group}\n"
        f"Cutout={CUTOUT_KIND}   N_Zern={N_ZERN}   solid=isDipole   dashed=non-dipole",
        fontsize=9,
        y=1.01,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"zernike_spectrum_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)

---
## 10. Figure B — Per-visit Zernike coefficient bar chart grid

In [ ]:
_ZERN_BAR_COLORS = {
    1: "#b39ddb",
    2: "#ef9a9a",
    3: "#ef9a9a",
    4: "#90caf9",
    5: "#a5d6a7",
    6: "#a5d6a7",
    7: "#ffcc80",
    8: "#ffcc80",
    9: "#c8e6c9",
    10: "#c8e6c9",
    11: "#b0bec5",
}
DEFAULT_BAR_COLOR = "#e0e0e0"

for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    n_vis = len(band_res)
    n_cols = min(NCOLS_GRID, n_vis)
    n_rows = int(np.ceil(n_vis / n_cols))

    fig = plt.figure(figsize=(4.5 * n_cols, 4.2 * n_rows))
    fig.suptitle(
        f"Zernike c₁…c{N_DISPLAY} per visit — diaObjectId={DIAOBJECT_ID}   band={band}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}   (N_Zern={N_ZERN})",
        fontsize=12,
        color=bcolor,
        fontweight="bold",
        y=1.0,
    )

    all_band_coeffs = np.concatenate([r["z_coeffs"][:N_DISPLAY] for r in band_res])
    yabs = max(float(np.nanmax(np.abs(all_band_coeffs))), 1e-6) * 1.15

    for vis_idx, rec in enumerate(band_res):
        ax = fig.add_subplot(n_rows, n_cols, vis_idx + 1)
        n_disp = min(N_DISPLAY, len(rec["z_coeffs"]))
        jj_d = np.arange(1, n_disp + 1)
        cc_d = rec["z_coeffs"][:n_disp]
        bar_colors = [_ZERN_BAR_COLORS.get(j, DEFAULT_BAR_COLOR) for j in jj_d]

        ax.bar(jj_d, cc_d, color=bar_colors, edgecolor="k", linewidth=0.5, zorder=3)
        ax.axhline(0, color="k", lw=0.7, ls=":")
        ax.set_xticks(jj_d)
        ax.set_xticklabels(
            [ZERNIKE_LABELS.get(j, f"Z{j}").split("\n")[0] for j in jj_d], fontsize=7, rotation=45, ha="right"
        )
        ax.set_ylim(-yabs, yabs)
        ax.grid(True, axis="y", alpha=0.3)

        q = rec["quality_z"]
        dip_str = "  🔴" if rec["is_dipole"] else ""
        ax.set_title(
            f"visit {rec['visit']}  {rec['date']}{dip_str}\n"
            f"R²={q['r2']:.3f}  χ²r={q['chi2_reduced']:.3f}\n"
            f"|dip|={q['dipole_amplitude']:.3f}  θ={q['dipole_angle_deg']:.1f}°",
            fontsize=7,
        )

        # Dipole direction inset
        if N_ZERN >= 3:
            c2, c3 = float(rec["z_coeffs"][1]), float(rec["z_coeffs"][2])
            ax_ins = ax.inset_axes([0.75, 0.70, 0.22, 0.28])
            ax_ins.set_aspect("equal")
            ax_ins.set_xlim(-1, 1)
            ax_ins.set_ylim(-1, 1)
            ax_ins.axhline(0, color="#aaa", lw=0.5)
            ax_ins.axvline(0, color="#aaa", lw=0.5)
            amp_n = np.hypot(c2, c3)
            scale = min(amp_n / max(yabs / np.sqrt(2), 1e-9), 1.0)
            if amp_n > 1e-9:
                ax_ins.annotate(
                    "",
                    xy=(c2 / amp_n * scale, c3 / amp_n * scale),
                    xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color="#cc3333", lw=1.5),
                )
            ax_ins.add_patch(plt.Circle((0, 0), 1, color="#ddd", fill=False, lw=0.5))
            ax_ins.set_xticks([])
            ax_ins.set_yticks([])
            ax_ins.set_title("dip", fontsize=8, pad=1)

    for extra in range(n_vis, n_rows * n_cols):
        fig.add_subplot(n_rows, n_cols, extra + 1).axis("off")

    plt.tight_layout()
    if SAVE_FIGS:
        stem = f"zernike_bars_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)

---
## 11. Figure C — Image triplet: original / selected model / residuals

The model displayed here is controlled by the `SELECTED_METHOD` flag set in
section 1.  Change it and re-run this cell to compare methods visually.

In [ ]:
CMAP_DIFF = "RdBu_r"

recon_key, resid_key, quality_key, amp_key, angle_key = METHOD_KEYS[SELECTED_METHOD]
print(f"Displaying method: {SELECTED_METHOD}  (recon={recon_key}, residual={resid_key})")

for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    n_vis = len(band_res)

    all_orig = np.concatenate([r["arr"].ravel() for r in band_res])
    vmax_global = symvlim(all_orig[np.isfinite(all_orig)], DIFF_PERCENTILE)

    fig, axes = plt.subplots(
        n_vis,
        3,
        figsize=(12, 4.25 * n_vis),
        gridspec_kw={"hspace": 0.05, "wspace": 0.05},
        layout="constrained",
    )
    if n_vis == 1:
        axes = [axes]  # ensure 2-D indexing

    for vis_idx, rec in enumerate(band_res):
        ax_orig = axes[vis_idx][0]
        ax_recon = axes[vis_idx][1]
        ax_resid = axes[vis_idx][2]

        arr = rec["arr"]
        recon = rec[recon_key]
        residual = rec[resid_key]
        q = rec[quality_key]
        amp = rec[amp_key]
        angle = rec[angle_key]

        vmax_res = symvlim(residual[np.isfinite(residual)], DIFF_PERCENTILE)
        dip_str = "  🔴 DIPOLE" if rec["is_dipole"] else ""

        for ax, img, vmax_img, title_suffix in [
            (ax_orig, arr, vmax_global, f"Original D\n(±{vmax_global:.1f})"),
            (ax_recon, recon, vmax_global, f"{SELECTED_METHOD} model\n(±{vmax_global:.1f})"),
            (ax_resid, residual, vmax_res, f"Residual D − model\n(±{vmax_res:.1f})"),
        ]:
            disp = np.where(np.isfinite(img), img, 0.0)
            im = ax.imshow(disp, origin="lower", cmap=CMAP_DIFF, vmin=-vmax_img, vmax=vmax_img)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            draw_crosshair(ax, disp.shape)
            ax.axis("off")
            ax.set_title(title_suffix, fontsize=8)

        row_label = (
            f"visit {rec['visit']}  {rec['date']}{dip_str}\n"
            f"R²={q['r2']:.3f}  χ²r={q['chi2_reduced']:.3f}  "
            f"A={amp:.2f}  θ={angle:.1f}°"
        )
        ax_orig.set_ylabel(row_label, fontsize=7.5, labelpad=4)
        ax_orig.axis("on")
        ax_orig.set_yticks([])
        ax_orig.set_xticks([])
        for sp in ax_orig.spines.values():
            sp.set_visible(False)

    fig.suptitle(
        f"Method: {SELECTED_METHOD} — diaObjectId={DIAOBJECT_ID}   band={band}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}   cutout={CUTOUT_KIND}",
        fontsize=10,
        color=bcolor,
        fontweight="bold",
        y=1.00,
    )

    if SAVE_FIGS:
        stem = f"dipole_fit_{SELECTED_METHOD}_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)

---
## 12. Figure D — Dipole amplitude & angle evolution across visits

Shows the evolution over time for the selected method and optionally overlays
all methods for comparison.

In [ ]:
fig, (ax_amp, ax_ang) = plt.subplots(2, 1, figsize=(11, 6), sharex=False)

for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    mjds = np.array([r["mjd"] for r in band_res])
    amps = np.array([r[amp_key] for r in band_res])
    angles = np.array([r[angle_key] for r in band_res])
    is_dip = np.array([r["is_dipole"] for r in band_res])
    dt = mjds - mjds.min()

    ax_amp.scatter(dt[~is_dip], amps[~is_dip], color=bcolor, marker="o", s=40, alpha=0.7, label=band)
    ax_amp.scatter(
        dt[is_dip],
        amps[is_dip],
        color=bcolor,
        marker="*",
        s=180,
        edgecolors="k",
        lw=0.8,
        alpha=0.95,
        label=f"{band} isDipole",
    )
    ax_amp.plot(dt, amps, color=bcolor, lw=0.8, alpha=0.5)

    ax_ang.scatter(dt[~is_dip], angles[~is_dip], color=bcolor, marker="o", s=40, alpha=0.7)
    ax_ang.scatter(
        dt[is_dip], angles[is_dip], color=bcolor, marker="*", s=180, edgecolors="k", lw=0.8, alpha=0.95
    )
    ax_ang.plot(dt, angles, color=bcolor, lw=0.8, alpha=0.5)

ax_amp.set_ylabel("Dipole amplitude", fontsize=9)
ax_amp.set_title(f"Dipole amplitude vs time — method={SELECTED_METHOD}", fontsize=10)
ax_amp.legend(fontsize=8, ncol=4, framealpha=0.8)
ax_amp.grid(True, alpha=0.3)

ax_ang.axhline(0, color="k", lw=0.5, ls=":")
ax_ang.set_ylabel("Dipole angle [°]", fontsize=9)
ax_ang.set_xlabel("Δt [days from first detection]", fontsize=9)
ax_ang.set_title(f"Dipole angle vs time — method={SELECTED_METHOD}", fontsize=10)
ax_ang.set_ylim(-185, 185)
ax_ang.grid(True, alpha=0.3)

fig.suptitle(
    f"diaObjectId={DIAOBJECT_ID}   Gaia G={G_str}  status={gaia_status}  group={fink_group}\n"
    f"Dipole evolution — method={SELECTED_METHOD}   ★ = isDipole flag",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()

if SAVE_FIGS:
    stem = f"dipole_evolution_{SELECTED_METHOD}_obj{DIAOBJECT_ID}"
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
    print(f"→ saved {stem}.{{pdf,png}}")
plt.show()
plt.close(fig)

---
## 13. Figure E — Zernike group-power bar chart per band

In [ ]:
GROUP_COLORS = {
    "Z1_piston": "#b39ddb",
    "Z2-3_dipole": "#ef5350",
    "Z4_defocus": "#42a5f5",
    "Z5-6_astig": "#66bb6a",
    "Z7-8_coma": "#ffa726",
    "Z9-10_trefoil": "#26c6da",
    "Z11_spher": "#8d6e63",
    "Z12+_higher": "#bdbdbd",
}

bands_with_results = [b for b in bands_to_use if results.get(b)]
if not bands_with_results:
    print("No results to display.")
else:
    n_bands = len(bands_with_results)
    fig, axes = plt.subplots(1, n_bands, figsize=(4.5 * n_bands, 5), squeeze=False)

    for ax_i, band in enumerate(bands_with_results):
        band_res = results[band]
        bcolor = BAND_COLORS.get(band, "gray")
        ax = axes[0][ax_i]
        group_names = list(GROUP_COLORS.keys())
        total_power = {g: 0.0 for g in group_names}
        for rec in band_res:
            for g, v in rec["z_power"].items():
                if g in total_power:
                    total_power[g] += v
        powers = np.array([total_power[g] for g in group_names])
        fracs = 100.0 * powers / max(powers.sum(), 1e-30)
        colors = [GROUP_COLORS[g] for g in group_names]
        labels = [g.split("_")[1] for g in group_names]
        bars = ax.bar(labels, fracs, color=colors, edgecolor="k", linewidth=0.6)
        ax.set_ylim(0, max(fracs) * 1.25)
        ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_title(f"Band {band}\n{len(band_res)} visits", fontsize=10, color=bcolor, fontweight="bold")
        for bar, frac in zip(bars, fracs):
            if frac > 1.0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{frac:.1f}%",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )

    fig.suptitle(
        f"Zernike group power — diaObjectId={DIAOBJECT_ID}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}   "
        f"cutout={CUTOUT_KIND}   N_Zern={N_ZERN}",
        fontsize=10,
        y=1.02,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"zernike_group_power_obj{DIAOBJECT_ID}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)

---
## 14. Final per-visit summary table (selected method)

Detailed table for the method selected by `SELECTED_METHOD`, including
all key fit metrics and Zernike coefficients c1…cN_DISPLAY.

In [ ]:
recon_key, resid_key, quality_key, amp_key, angle_key = METHOD_KEYS[SELECTED_METHOD]

rows_final = []
for band in bands_to_use:
    for rec in results.get(band, []):
        q = rec[quality_key]
        row = {
            "band": band,
            "visit": rec["visit"],
            "date": rec["date"],
            "isDipole": rec["is_dipole"],
            "psfFlux_nJy": round(rec["psfFlux"], 2),
            "snr": round(rec["snr"], 2) if np.isfinite(rec["snr"]) else np.nan,
            "method": SELECTED_METHOD,
            "R2": round(float(q.get("r2", np.nan)), 4),
            "chi2_reduced": round(float(q.get("chi2_reduced", np.nan)), 4),
            "rms_input": round(float(q.get("rms_input", np.nan)), 4),
            "rms_residual": round(float(q.get("rms_residual", np.nan)), 4),
            "dipole_amp": round(rec[amp_key], 4),
            "dipole_angle": round(rec[angle_key], 2),
        }
        # Append Zernike coefficients if available
        if "z_coeffs" in rec:
            for j in range(1, min(N_DISPLAY, len(rec["z_coeffs"])) + 1):
                row[f"c{j}"] = round(float(rec["z_coeffs"][j - 1]), 5)
        rows_final.append(row)

df_final = pd.DataFrame(rows_final)
print(f"Summary — method={SELECTED_METHOD}   diaObjectId={DIAOBJECT_ID}")
ipy_display(df_final)

csv_path = os.path.join(DIR_FIGS, f"summary_{SELECTED_METHOD}_obj{DIAOBJECT_ID}.csv")
df_final.to_csv(csv_path, index=False)
print(f"→ Saved to {csv_path}")